# 01 Data Collection and Cleaning

This notebook downloads historical daily cryptocurrency market data, performs initial cleaning, checks basic data quality, and stores both a full historical dataset and an aligned modeling dataset.

Two dataset versions are created:

- Full dataset: preserves the full history from 2017 onward, even if some assets were not yet available during the early period
- Aligned dataset: only retains dates for which all selected assets have data

This distinction is important because some assets, such as Solana, have a shorter trading history than Bitcoin and the other selected altcoins. The full dataset is useful for exploratory analysis, while the aligned dataset is used for methods that require complete feature vectors, such as clustering, HMMs, and LSTM models.

In [1]:
from pathlib import Path
from typing import Dict

import pandas as pd
import yfinance as yf

## Configuration

This section defines the project paths, selected assets, and download settings.

### Output file naming convention
Processed datasets follow the pattern: `dataset_format_metric_scope`

| Chunk | Meaning                                                                 |
| --- |-------------------------------------------------------------------------|
| **dataset** | data source used in this project (e.g. crypto, macro)                   |
| **format** |                                                                         |
| `long` | one row per asset and date                                              |
| `wide` | one column per asset                                                    |
| **metric** |                                                                         |
| `close` | closing price                                                           |
| `volume` | traded volume                                                           |
| **scope** |                                                                         |
| `raw` | direct download from the data source                                    |
| `clean` | cleaned long-format dataset                                             |
| `full` | full historical dataset (may contain missing values for younger assets) |
| `aligned` | dataset restricted to dates where all assets are available              |

Additional files store data quality summaries and asset availability diagnostics.

In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Asset selection
TICKERS: Dict[str, str] = {
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
    "SOL": "SOL-USD",
    "XRP": "XRP-USD",
    "BNB": "BNB-USD",
    "TRX": "TRX-USD",
}

START_DATE = "2017-01-01"
END_DATE = None
INTERVAL = "1d"

# Raw data
RAW_OUTPUT_PATH = DATA_RAW_DIR / "crypto_long_raw.csv"

# Clean datasets
CLEAN_LONG_OUTPUT_PATH = DATA_PROCESSED_DIR / "crypto_long_clean.csv"

# Full historical wide datasets
FULL_CLOSE_OUTPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_close_full.csv"
FULL_VOLUME_OUTPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_volume_full.csv"

# Aligned modeling datasets
ALIGNED_CLOSE_OUTPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_close_aligned.csv"
ALIGNED_VOLUME_OUTPUT_PATH = DATA_PROCESSED_DIR / "crypto_wide_volume_aligned.csv"

# Diagnostics
QUALITY_SUMMARY_OUTPUT_PATH = DATA_PROCESSED_DIR / "data_quality_summary.csv"
AVAILABILITY_MATRIX_OUTPUT_PATH = DATA_PROCESSED_DIR / "asset_availability_matrix.csv"

## Download raw market data

The following function downloads daily **OHLCV** data for each selected asset and stores the result in a consistent tabular format.

| Indicator | Meaning                                            |
| --- |----------------------------------------------------|
| **Open** | The first traded price at the beginning of the day |
| **High** | The highest traded price for the day               |
| **Low** | The lowest traded price for the day                |
| **Close** | The final traded price at the end of the day       |
| **Volume** | The number of coins traded in the day              |

In [3]:
def download_single_ticker(ticker: str, symbol: str, start_date: str, end_date: str | None, interval: str) -> pd.DataFrame:
    """
    Download daily market data for a single ticker from yfinance and return a cleaned raw DataFrame.

    Parameters
    ----------
    ticker : str
        Internal short name, e.g. BTC.
    symbol : str
        Yahoo Finance ticker symbol, e.g. BTC-USD.
    start_date : str
        Download start date in YYYY-MM-DD format.
    end_date : str | None
        Download end date in YYYY-MM-DD format. If None, yfinance uses the latest available date.
    interval : str
        Data interval, e.g. '1d'.

    Returns
    -------
    pd.DataFrame
        DataFrame with one row per date and standard OHLCV columns.
    """
    df = yf.download(
        symbol,
        start=start_date,
        end=end_date,
        interval=interval,
        auto_adjust=False,
        progress=False,
    )

    if df.empty:
        raise ValueError(f"No data returned for {ticker} ({symbol}).")

    df = df.reset_index()

    # Flatten possible multi-index columns returned by yfinance
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]

    expected_columns = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]
    available_columns = [col for col in expected_columns if col in df.columns]
    df = df[available_columns].copy()

    df["Ticker"] = ticker
    df["Symbol"] = symbol
    df["Date"] = pd.to_datetime(df["Date"])

    return df

In [4]:
raw_frames = []

for ticker, symbol in TICKERS.items():
    df_ticker = download_single_ticker(
        ticker=ticker,
        symbol=symbol,
        start_date=START_DATE,
        end_date=END_DATE,
        interval=INTERVAL,
    )
    raw_frames.append(df_ticker)

raw_df = pd.concat(raw_frames, ignore_index=True)
raw_df = raw_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

raw_df.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Ticker,Symbol
0,2017-11-09,2.05314,2.17423,1.89394,1.99077,1.99077,19192200,BNB,BNB-USD
1,2017-11-10,2.00773,2.06947,1.64478,1.79684,1.79684,11155000,BNB,BNB-USD
2,2017-11-11,1.78628,1.91775,1.61429,1.67047,1.67047,8178150,BNB,BNB-USD
3,2017-11-12,1.66889,1.67280,1.46256,1.51969,1.51969,15298700,BNB,BNB-USD
4,2017-11-13,1.52601,1.73502,1.51760,1.68662,1.68662,12238800,BNB,BNB-USD


## Save raw combined dataset

The raw combined dataset is stored before further cleaning so that the original download can be reproduced and inspected later.

In [5]:
raw_df.to_csv(RAW_OUTPUT_PATH, index=False)
print(f"Saved raw dataset to: {RAW_OUTPUT_PATH}")

Saved raw dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\raw\crypto_long_raw.csv


## Basic raw data inspection

This section checks the structure and completeness of the downloaded dataset.

In [6]:
print(raw_df.info())
print()
print(raw_df.describe(include="all"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17771 entries, 0 to 17770
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       17771 non-null  datetime64[ns]
 1   Open       17771 non-null  float64       
 2   High       17771 non-null  float64       
 3   Low        17771 non-null  float64       
 4   Close      17771 non-null  float64       
 5   Adj Close  17771 non-null  float64       
 6   Volume     17771 non-null  int64         
 7   Ticker     17771 non-null  object        
 8   Symbol     17771 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(2)
memory usage: 1.2+ MB
None

                                 Date           Open           High  \
count                           17771   17771.000000   17771.000000   
unique                            NaN            NaN            NaN   
top                               NaN            NaN            NaN   
freq

In [7]:
raw_df.isna().sum()

Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
Ticker       0
Symbol       0
dtype: int64

In [8]:
raw_df.duplicated(subset=["Ticker", "Date"]).sum()

np.int64(0)

## Data quality summary by asset

The following summary provides a first overview of the available history and missing values for each asset.

In [9]:
summary_dict = {
    "symbol": ("Symbol", "first"),
    "first_date": ("Date", "min"),
    "last_date": ("Date", "max"),
    "n_rows": ("Date", "count"),
    "missing_open": ("Open", lambda s: s.isna().sum()),
    "missing_high": ("High", lambda s: s.isna().sum()),
    "missing_low": ("Low", lambda s: s.isna().sum()),
    "missing_close": ("Close", lambda s: s.isna().sum()),
    "missing_volume": ("Volume", lambda s: s.isna().sum()),
}

if "Adj Close" in raw_df.columns:
    summary_dict["missing_adj_close"] = ("Adj Close", lambda s: s.isna().sum())

quality_summary = (
    raw_df
    .groupby("Ticker")
    .agg(**summary_dict)
    .reset_index()
)

quality_summary

,Ticker,symbol,first_date,last_date,n_rows,missing_open,missing_high,missing_low,missing_close,missing_volume,missing_adj_close
0,BNB,BNB-USD,2017-11-09,2026-03-23,3057,0,0,0,0,0,0
1,BTC,BTC-USD,2017-01-01,2026-03-23,3369,0,0,0,0,0,0
2,ETH,ETH-USD,2017-11-09,2026-03-23,3057,0,0,0,0,0,0
3,SOL,SOL-USD,2020-04-10,2026-03-23,2174,0,0,0,0,0,0
4,TRX,TRX-USD,2017-11-09,2026-03-23,3057,0,0,0,0,0,0
5,XRP,XRP-USD,2017-11-09,2026-03-23,3057,0,0,0,0,0,0


In [10]:
quality_summary.to_csv(QUALITY_SUMMARY_OUTPUT_PATH, index=False)
print(f"Saved quality summary to: {QUALITY_SUMMARY_OUTPUT_PATH}")

Saved quality summary to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\data_quality_summary.csv


## Keep relevant columns

The most relevant columns for our project are date, close price, and volume. The full OHLCV data remains available in the raw export if needed later.

In [11]:
clean_long_df = raw_df[["Date", "Ticker", "Symbol", "Close", "Volume"]].copy()
clean_long_df = clean_long_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

clean_long_df.head()

,Date,Ticker,Symbol,Close,Volume
0,2017-11-09,BNB,BNB-USD,1.99077,19192200
1,2017-11-10,BNB,BNB-USD,1.79684,11155000
2,2017-11-11,BNB,BNB-USD,1.67047,8178150
3,2017-11-12,BNB,BNB-USD,1.51969,15298700
4,2017-11-13,BNB,BNB-USD,1.68662,12238800


## Create wide-format datasets

Two wide-format versions are created, a full and an aligned dataset.

In [12]:
close_wide_full_df = clean_long_df.pivot(index="Date", columns="Ticker", values="Close").sort_index()
volume_wide_full_df = clean_long_df.pivot(index="Date", columns="Ticker", values="Volume").sort_index()

print("Full close shape:", close_wide_full_df.shape)
print("Full volume shape:", volume_wide_full_df.shape)

Full close shape: (3369, 6)
Full volume shape: (3369, 6)


### Remove rows without any available data

Some of the earliest dates returned by the download may contain no observations for any of the selected assets. These rows do not contain useful information and are removed.

Rows are only dropped when **all assets are missing**, so the full dataset still preserves the early history of assets that started trading earlier than others.

In [13]:
close_wide_full_df = close_wide_full_df.dropna(how="all")
volume_wide_full_df = volume_wide_full_df.dropna(how="all")

## Asset availability over time

The following matrix indicates whether price data is available for each asset on each date. This helps identify the start date of younger assets such as Solana.

In [14]:
availability_matrix = close_wide_full_df.notna().astype(int)
availability_matrix.head()

Ticker,BNB,BTC,ETH,SOL,TRX,XRP
Date,,,,,,
2017-01-01,0,1,0,0,0,0
2017-01-02,0,1,0,0,0,0
2017-01-03,0,1,0,0,0,0
2017-01-04,0,1,0,0,0,0
2017-01-05,0,1,0,0,0,0


In [15]:
availability_matrix.to_csv(AVAILABILITY_MATRIX_OUTPUT_PATH, index=True)
print(f"Saved availability matrix to: {AVAILABILITY_MATRIX_OUTPUT_PATH}")

Saved availability matrix to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\asset_availability_matrix.csv


In [16]:
asset_start_dates = close_wide_full_df.apply(lambda col: col.first_valid_index()).to_frame(name="first_valid_date")
asset_end_dates = close_wide_full_df.apply(lambda col: col.last_valid_index()).to_frame(name="last_valid_date")

availability_summary = asset_start_dates.join(asset_end_dates)
availability_summary

,first_valid_date,last_valid_date
Ticker,,
BNB,2017-11-09,2026-03-23
BTC,2017-01-01,2026-03-23
ETH,2017-11-09,2026-03-23
SOL,2020-04-10,2026-03-23
TRX,2017-11-09,2026-03-23
XRP,2017-11-09,2026-03-23


## Create aligned modeling dataset

The aligned dataset only keeps dates where all selected assets have valid observations. This is the version used later for modeling steps that require complete feature vectors.

In [17]:
close_wide_aligned_df = close_wide_full_df.dropna(how="any").copy()
volume_wide_aligned_df = volume_wide_full_df.loc[close_wide_aligned_df.index].copy()

print("Aligned close shape:", close_wide_aligned_df.shape)
print("Aligned volume shape:", volume_wide_aligned_df.shape)

Aligned close shape: (2174, 6)
Aligned volume shape: (2174, 6)


In [18]:
print("Aligned start date:", close_wide_aligned_df.index.min())
print("Aligned end date:", close_wide_aligned_df.index.max())

Aligned start date: 2020-04-10 00:00:00
Aligned end date: 2026-03-23 00:00:00


## Final checks

These checks confirm that the cleaned aligned datasets no longer contain missing values.

In [19]:
print("Missing values in full close dataset:")
print(close_wide_full_df.isna().sum())

print()
print("Missing values in aligned close dataset:")
print(close_wide_aligned_df.isna().sum())

Missing values in full close dataset:
Ticker
BNB     312
BTC       0
ETH     312
SOL    1195
TRX     312
XRP     312
dtype: int64

Missing values in aligned close dataset:
Ticker
BNB    0
BTC    0
ETH    0
SOL    0
TRX    0
XRP    0
dtype: int64


In [20]:
print("Missing values in full volume dataset:")
print(volume_wide_full_df.isna().sum())

print()
print("Missing values in aligned volume dataset:")
print(volume_wide_aligned_df.isna().sum())

Missing values in full volume dataset:
Ticker
BNB     312
BTC       0
ETH     312
SOL    1195
TRX     312
XRP     312
dtype: int64

Missing values in aligned volume dataset:
Ticker
BNB    0
BTC    0
ETH    0
SOL    0
TRX    0
XRP    0
dtype: int64


## Save cleaned outputs

The cleaned outputs are saved in both long and wide format for later notebooks.

In [21]:
clean_long_df.to_csv(CLEAN_LONG_OUTPUT_PATH, index=False)

close_wide_full_df.to_csv(FULL_CLOSE_OUTPUT_PATH, index=True)
volume_wide_full_df.to_csv(FULL_VOLUME_OUTPUT_PATH, index=True)

close_wide_aligned_df.to_csv(ALIGNED_CLOSE_OUTPUT_PATH, index=True)
volume_wide_aligned_df.to_csv(ALIGNED_VOLUME_OUTPUT_PATH, index=True)

print(f"Saved clean long dataset to: {CLEAN_LONG_OUTPUT_PATH}")
print(f"Saved full close dataset to: {FULL_CLOSE_OUTPUT_PATH}")
print(f"Saved full volume dataset to: {FULL_VOLUME_OUTPUT_PATH}")
print(f"Saved aligned close dataset to: {ALIGNED_CLOSE_OUTPUT_PATH}")
print(f"Saved aligned volume dataset to: {ALIGNED_VOLUME_OUTPUT_PATH}")

Saved clean long dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\crypto_long_clean.csv
Saved full close dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\crypto_wide_close_full.csv
Saved full volume dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\crypto_wide_volume_full.csv
Saved aligned close dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\crypto_wide_close_aligned.csv
Saved aligned volume dataset to: C:\Users\taula\Github\HSLU\DSPRO2\HSLU_FS25_DSPRO2\data\processed\crypto_wide_volume_aligned.csv
